# RAG Q&A on PDFs with Enhanced Embeddings

In [1]:
!pip install transformers
!pip install langchain[docarray]
!pip install docarray
!pip install pypdf
!pip install langchain_huggingface
!pip install bitsandbytes
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained("unsloth/gemma-2-9b-it", use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    "unsloth/gemma-2-9b-it",
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=64
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device set to use cuda:0


In [3]:
# Load the new pdf file
!gdown --id 1P_K5J4zsKpaxDw9lphU15X18qb5VcSdQ -O original.pdf
# Rename the downloaded file
!mv original.pdf new_doc.pdf

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1P_K5J4zsKpaxDw9lphU15X18qb5VcSdQ
To: /content/original.pdf
100% 325k/325k [00:00<00:00, 35.0MB/s]


In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("new_doc.pdf")
pages = loader.load_and_split()
print(len(pages))


31


In [5]:
from langchain_huggingface.llms import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=text_gen)

def apply_chat_template_and_response(prompt):
    messages = [
    {'role': 'user', 'content': prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    return llm.invoke(text).replace(text, '')

In [6]:
# from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate

template = """
You are a helpful and knowledgeable AI assistant. Use only the information retrieved from the documents to answer the user's question in Persian (Farsi). If the answer is not found in the retrieved context, respond with: "متاسفانه اطلاعاتی در این مورد ندارم." Do not use your own knowledge beyond the provided context. Be accurate, clear, and polite. Never mention the documents or the retrieval process in your response. Just respond naturally in Persian.
Context: {context}

Question: {question}

Answer:

"""

prompt = PromptTemplate.from_template(template)
prompt.format(context="Here is some context", question="Here is a question")

'\nYou are a helpful and knowledgeable AI assistant. Use only the information retrieved from the documents to answer the user\'s question in Persian (Farsi). If the answer is not found in the retrieved context, respond with: "متاسفانه اطلاعاتی در این مورد ندارم." Do not use your own knowledge beyond the provided context. Be accurate, clear, and polite. Never mention the documents or the retrieval process in your response. Just respond naturally in Persian.\nContext: Here is some context\n\nQuestion: Here is a question\n\nAnswer:\n\n'

In [7]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=256)
# text_documents = text_splitter.split_documents(pages)[:5]

pages

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2019-07-23T14:02:50+04:30', 'title': 'Slide 1', 'author': 'Nafiseh Hosseini', 'moddate': '2019-07-23T14:02:50+04:30', 'source': 'new_doc.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1'}, page_content='قاچاق در حوزه\nتجهیزات و ملزومات پزشکی\nارائه از :\nمجید حمیدی'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2019-07-23T14:02:50+04:30', 'title': 'Slide 1', 'author': 'Nafiseh Hosseini', 'moddate': '2019-07-23T14:02:50+04:30', 'source': 'new_doc.pdf', 'total_pages': 31, 'page': 1, 'page_label': '2'}, page_content='رئوسمطالب:\n\uf0a7مقدمه\n\uf0a7اهمیتقاچاقدرحوزهتجهیزاتوملزوماتپزشکی\n\uf0a7اقداماتانجامشدهتوسطادارهکلتجهیزاتپزشکیدرحوزهنظارتها\n\uf0a7مروریبرموادمهمقانونمبارزهباقاچاقکالاوارزدرحوزهتجهیزاتو\nملزوماتپزشکی'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creato

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=256
)

docs = splitter.split_documents(pages)
len(docs)


31

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings


embedding_name_1 = 'sbunlp/fabert'
embeddings_1 = HuggingFaceEmbeddings(
    model_name=embedding_name_1,
    # model_kwargs={"trust_remote_code": True},
)

config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at sbunlp/fabert and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [10]:
from langchain_community.vectorstores import DocArrayInMemorySearch

In [11]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [13]:
vectorstore_1 = DocArrayInMemorySearch.from_documents(docs, embedding=embeddings_1)
retriever_1= vectorstore_1.as_retriever()

In [15]:
query = "ملزومات پزشکی"
retriever = vectorstore_1.as_retriever()
retriever.invoke(query)


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2019-07-23T14:02:50+04:30', 'title': 'Slide 1', 'author': 'Nafiseh Hosseini', 'moddate': '2019-07-23T14:02:50+04:30', 'source': 'new_doc.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1'}, page_content='قاچاق در حوزه\nتجهیزات و ملزومات پزشکی\nارائه از :\nمجید حمیدی'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2019-07-23T14:02:50+04:30', 'title': 'Slide 1', 'author': 'Nafiseh Hosseini', 'moddate': '2019-07-23T14:02:50+04:30', 'source': 'new_doc.pdf', 'total_pages': 31, 'page': 1, 'page_label': '2'}, page_content='رئوسمطالب:\n\uf0a7مقدمه\n\uf0a7اهمیتقاچاقدرحوزهتجهیزاتوملزوماتپزشکی\n\uf0a7اقداماتانجامشدهتوسطادارهکلتجهیزاتپزشکیدرحوزهنظارتها\n\uf0a7مروریبرموادمهمقانونمبارزهباقاچاقکالاوارزدرحوزهتجهیزاتو\nملزوماتپزشکی'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creato

In [ ]:
retrieved_context = retriever_1.invoke(query)


In [27]:
questions = [
    "این فایل درباره چیست؟",
    "چه نکات مهمی در این سند مطرح شده است؟",
    "هدف اصلی نویسندگان از تهیه این فایل چیست؟"
]

for question in questions:
    # Retrieve context from the vector store
    retrieved_context = retriever_1.invoke(question)

    # Join the content of retrieved documents
    context = "\n".join([d.page_content for d in retrieved_context])

    # Format the prompt with the retrieved context and the question
    formatted_prompt = prompt.format(context=context, question=question)

    # Generate response using your LLM function
    response_from_model = apply_chat_template_and_response(formatted_prompt)

    # Parse the response if you have a parser
    parsed_response = parser.parse(response_from_model)

    # Print question and answer
    print("🔹 سوال:", question)
    print("🔸 پاسخ:", parsed_response)
    print("-" * 70)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🔹 سوال: این فایل درباره چیست؟
🔸 پاسخ: این فایل در مورد قانون مبارزه با قاچاق کالا و ارز، به ویژه در حوزه تجهیزات و ملزومات پزشکی و اقدامات انجام شده توسط اداره کل تجهیزات پزشکی در این زمینه صحبت می کند.  


----------------------------------------------------------------------
🔹 سوال: چه نکات مهمی در این سند مطرح شده است؟
🔸 پاسخ: در این سند  در خصوص شناسه کالا و شناسه رهگیری توضیحاتی ارائه شده است. شناسه کالا شامل چند رقم است و ماهیت کالا را مشخص می کند و به صورت رمزنگاری شده بر روی کالا نصب یا درج می شود. شناسه رهگیری نیز چند رقمی است و برای منح
----------------------------------------------------------------------
🔹 سوال: هدف اصلی نویسندگان از تهیه این فایل چیست؟
🔸 پاسخ: متاسفانه اطلاعاتی در این مورد ندارم. 



----------------------------------------------------------------------


In [17]:
questions = [
    "تعریف قاچاق کالا و ارز"
]

for question in questions:
    retrieved_context = retriever_1.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: تعریف قاچاق کالا و ارز
Answer: قاچاق کالا و ارز، جرمی است که در آن کالایی یا ارز به صورت غیرمجاز وارد یا خارج از کشور می‌شود. 






In [18]:
questions = [
"دلایل اهمیت نظارت و برخورد با قاچاق "
]

for question in questions:
    retrieved_context = retriever_1.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: دلایل اهمیت نظارت و برخورد با قاچاق 
Answer: متاسفانه اطلاعاتی در این مورد ندارم. 




In [19]:
questions = [
"پیامدهای اقتصادی ناشی از قاچاق"
]

for question in questions:
    retrieved_context = retriever_1.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: پیامدهای اقتصادی ناشی از قاچاق
Answer: کاهش درآمدهای دولت( عوارض گمرکی و درآمدهای ارزی)
کاهش سطح تولید ملی
کاهش اشتغال
هدر رفتن منابع ارزی کشور 




In [26]:
questions = [
"رئوس مطالب"
]

for question in questions:
    retrieved_context = retriever_1.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: رئوس مطالب
Answer: * مقدمه 
* اهمیت قاچاق در حوزه تجهیزات و ملزومات پزشکی
* اقدامات انجام شده توسط اداره کل تجهیزات پزشکی در حوزه نظارت ها
* مروری بر موضوع مهم قانون مبارزه با قاچاق کالا و ارز در حوزه تجهیزات و ملزومات پزشکی 






In the first parT, I used the embedding model **sbunlp/fabert** to convert the PDF text into vector representations.

To improve the retrieval quality, I replaced the embedding model with **sentence-transformers/paraphrase-multilingual-mpnet-base-v2**. This model is multilingual and optimized for semantic similarity tasks.By using this model:

•	The retriever can find more relevant passages even if the exact words do not match the query.
•	Answers became more accurate and contextually appropriate.

After switching to this model (embedding_name_2 = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"), I reran the same three questions. The quality of the retrieved context improved, and the answers generated by the LLM were more precise and relevant.



In [20]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 39.3 MB/s eta 0:00:00


In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_name_2 = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embeddings_2 = HuggingFaceEmbeddings(model_name= embedding_name_2)

vectorstore_2 = FAISS.from_documents(docs, embeddings_2)
retriever_2 = vectorstore_2.as_retriever(search_kwargs={"k": 4})


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [28]:
questions = [
    "این فایل درباره چیست؟",
    "چه نکات مهمی در این سند مطرح شده است؟",
    "هدف اصلی نویسندگان از تهیه این فایل چیست؟"
]

for question in questions:
    # Retrieve context from the vector store
    retrieved_context = retriever_1.invoke(question)

    # Join the content of retrieved documents
    context = "\n".join([d.page_content for d in retrieved_context])

    # Format the prompt with the retrieved context and the question
    formatted_prompt = prompt.format(context=context, question=question)

    # Generate response using your LLM function
    response_from_model = apply_chat_template_and_response(formatted_prompt)

    # Parse the response if you have a parser
    parsed_response = parser.parse(response_from_model)

    # Print question and answer
    print("🔹 سوال:", question)
    print("🔸 پاسخ:", parsed_response)
    print("-" * 70)


🔹 سوال: این فایل درباره چیست؟
🔸 پاسخ: این فایل درباره مقررات مربوط به دستگاه کاشف و کشف کالا در حوزه قاچاق تجهیزات و ملزومات پزشکی است.  

----------------------------------------------------------------------
🔹 سوال: چه نکات مهمی در این سند مطرح شده است؟
🔸 پاسخ: در این سند موارد زیر مورد بحث قرار گرفته است:

* تعریف قاچاق کالا و ارز
* مصادیق قاچاق کالا
* تشریفات قانونی مربوط به وارد و صادرات کالا و ارز
* دستگاه های کاشف قاچاق کالا
* دلایل اهمیت
----------------------------------------------------------------------
🔹 سوال: هدف اصلی نویسندگان از تهیه این فایل چیست؟
🔸 پاسخ: هدف اصلی نویسندگان از تهیه این فایل،  اهمیت نظارت و برخورد با قاچاق تجهیزات و ملزومات پزشکی را شرح دادن و دلایل آن را بیان کردن است. 



----------------------------------------------------------------------


In [22]:
questions = [
"پیامدهای اقتصادی ناشی از قاچاق"
]

for question in questions:
    retrieved_context = retriever_2.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: پیامدهای اقتصادی ناشی از قاچاق
Answer: کاهش درآمدهای دولت( عوارض گمرکی و درآمدهای ارزی)
کاهش سطح تولید ملی
کاهش اشتغال
هدر رفتن منابع ارزی کشور 




In [23]:
questions = [
    "تعریف قاچاق کالا و ارز"
]

for question in questions:
    retrieved_context = retriever_2.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: تعریف قاچاق کالا و ارز
Answer: هر فعل یا ترک فعلی است که موجب نقض تشریفات قانونی مربوط به ورود و خروج کالا و ارز گردد و براساس این قوانین و یا سایر قوانین، قاچاق محسوب و برای آن مجازات تعیین شده باشد، در اماکن ورودی، تمامی نقاط از کشور حتی



In [25]:
questions = [
"رئوس مطالب"
]

for question in questions:
    retrieved_context = retriever_2.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: رئوس مطالب
Answer: رئوسمطالب:

مقدمه
اهمیت قاچاق در حوزه تجهیزات و ملزومات پزشکی
مراجعه به اقدامات انجام شده توسط اداره کل تجهیزات پزشکی در حوزه نظارت ها
مروری بر ماده مهم قانون مبارزه با قاچاق کالا و ارز در حوزه تجهیزات و ملز



In [24]:
questions = [
"دلایل اهمیت نظارت و برخورد با قاچاق "
]

for question in questions:
    retrieved_context = retriever_2.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: دلایل اهمیت نظارت و برخورد با قاچاق 
Answer: متاسفانه اطلاعاتی در این مورد ندارم. 




In [ ]:
questions = [
"پیامدهای اقتصادی ناشی از قاچاق"
]

for question in questions:
    retrieved_context = retriever_2.invoke(question)
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = apply_chat_template_and_response(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()